# 📖 Notebook 3: Cron-Like Scheduling & Dead Letter Queues

In the previous notebooks, we built priority queues and worker pools. Now we tackle the remaining pieces: **recurring schedules** (cron jobs) and **dead letter queues** (DLQ) for permanently failed jobs.

## Learning Objectives

- Parse and evaluate cron expressions
- Generate the next execution time for recurring jobs
- Build a watcher that creates execution rows on schedule
- Implement a dead letter queue for failed jobs
- Understand the full end-to-end scheduling lifecycle

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/job-scheduler
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `job_scheduler`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import time
import uuid
import random
from datetime import datetime, timedelta
from croniter import croniter

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "job_scheduler",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

r = get_redis()

try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## ⏰ Understanding Cron Expressions

A cron expression is a compact way to describe a recurring schedule. It has **5 fields**:

```
┌───────────── minute (0 - 59)
│ ┌───────────── hour (0 - 23)
│ │ ┌───────────── day of month (1 - 31)
│ │ │ ┌───────────── month (1 - 12)
│ │ │ │ ┌───────────── day of week (0 - 6, 0 = Sunday)
│ │ │ │ │
* * * * *
```

Examples:
- `0 9 * * 1` → Every Monday at 9:00 AM
- `*/15 * * * *` → Every 15 minutes
- `0 2 * * *` → Every day at 2:00 AM
- `0 0 1 * *` → First day of every month at midnight
- `0 3 * * 0` → Every Sunday at 3:00 AM

We use the `croniter` library to parse these and compute the next execution time.

In [ ]:
# Understanding cron expressions with croniter

examples = [
    ("0 9 * * 1",     "Every Monday at 9:00 AM"),
    ("*/15 * * * *",  "Every 15 minutes"),
    ("0 2 * * *",     "Every day at 2:00 AM"),
    ("0 0 1 * *",     "First of every month at midnight"),
    ("0 3 * * 0",     "Every Sunday at 3:00 AM"),
    ("30 8 * * 1-5",  "Weekdays at 8:30 AM"),
]

now = datetime.now()
print(f"Current time: {now.strftime('%Y-%m-%d %H:%M:%S')}")
print()
print("📅 Cron Expression Examples")
print("=" * 75)

for expression, description in examples:
    cron = croniter(expression, now)
    next_run = cron.get_next(datetime)
    after_that = cron.get_next(datetime)
    
    print(f"  {expression:<20} {description}")
    print(f"  {'':20} Next:  {next_run.strftime('%Y-%m-%d %H:%M')}")
    print(f"  {'':20} After: {after_that.strftime('%Y-%m-%d %H:%M')}")
    print()

In [ ]:
# Let's look at the cron jobs in our database

conn = get_db()
cursor = conn.cursor()

cursor.execute("""
    SELECT j.id::text, j.user_id, t.name, j.schedule_value, j.parameters
    FROM jobs j
    JOIN tasks t ON j.task_id = t.id
    WHERE j.schedule_type = 'CRON' AND j.is_active = TRUE
    ORDER BY j.created_at
""")

print("📋 Active Cron Jobs in Database")
print("=" * 80)
now = datetime.now()

for row in cursor.fetchall():
    job_id, user_id, task_name, cron_expr, params = row
    cron = croniter(cron_expr, now)
    next_run = cron.get_next(datetime)
    
    print(f"  Job: {job_id[:8]}...")
    print(f"  User: {user_id}  |  Task: {task_name}")
    print(f"  Cron: {cron_expr}  |  Next run: {next_run.strftime('%Y-%m-%d %H:%M')}")
    print(f"  Params: {json.dumps(params)}")
    print()

conn.close()

## 🔄 The Watcher: Generating Executions for Cron Jobs

The watcher is the heart of the scheduling system. It runs periodically and does two things:

1. **For cron jobs**: compute the next execution time and create an execution row if one doesn't exist
2. **For all pending executions**: push due executions into the Redis queue

```
┌─────────────┐     creates        ┌──────────────┐     enqueues      ┌───────┐
│ Cron Jobs   │  ──────────────►  │  Executions  │  ──────────────►  │ Redis │
│ (recurring) │  next-run rows     │  (PENDING)   │  ready-to-run     │ Queue │
└─────────────┘                    └──────────────┘                   └───────┘
```

**Key insight**: We separate the *definition* (cron job) from the *instance* (execution). This way:
- Finding "what needs to run soon" is a simple timestamp comparison
- No need to evaluate every cron expression on every poll
- Each execution has its own status, retry count, and result

In [ ]:
# The watcher: create execution rows for upcoming cron runs

def schedule_next_cron_executions(look_ahead_minutes: int = 60):
    """For each active cron job, ensure an execution row exists for the next run.
    
    This is the first responsibility of the watcher process.
    We look ahead a configurable window and create PENDING executions.
    """
    conn = get_db()
    cursor = conn.cursor()
    
    # Find all active cron jobs
    cursor.execute("""
        SELECT id, schedule_value
        FROM jobs
        WHERE schedule_type = 'CRON' AND is_active = TRUE
    """)
    cron_jobs = cursor.fetchall()
    
    now = datetime.now()
    look_ahead = now + timedelta(minutes=look_ahead_minutes)
    created = 0
    
    for job_id, cron_expr in cron_jobs:
        cron = croniter(cron_expr, now)
        
        # Generate all runs within the look-ahead window
        while True:
            next_run = cron.get_next(datetime)
            if next_run > look_ahead:
                break
            
            # Check if an execution already exists for this time
            cursor.execute("""
                SELECT 1 FROM executions
                WHERE job_id = %s AND scheduled_at = %s
            """, (job_id, next_run))
            
            if not cursor.fetchone():
                # Create the execution row
                cursor.execute("""
                    INSERT INTO executions (job_id, status, scheduled_at)
                    VALUES (%s, 'PENDING', %s)
                """, (job_id, next_run))
                created += 1
    
    conn.commit()
    conn.close()
    return created

# Run the watcher scheduling step
created = schedule_next_cron_executions(look_ahead_minutes=60 * 24)  # 24 hours
print(f"📅 Created {created} new execution rows for upcoming cron runs")

# Show what was created
conn = get_db()
cursor = conn.cursor()
cursor.execute("""
    SELECT e.scheduled_at, e.status, t.name, j.schedule_value
    FROM executions e
    JOIN jobs j ON e.job_id = j.id
    JOIN tasks t ON j.task_id = t.id
    WHERE j.schedule_type = 'CRON'
      AND e.scheduled_at > NOW()
    ORDER BY e.scheduled_at
    LIMIT 15
""")

print()
print("📋 Upcoming Executions (next 15):")
print("=" * 70)
for row in cursor.fetchall():
    print(f"  {row[0].strftime('%Y-%m-%d %H:%M')}  [{row[1]}]  {row[2]}  ({row[3]})")

conn.close()

In [ ]:
# Phase 2 of the watcher: push due executions into Redis

QUEUE_KEY = "scheduler:queue"
r.delete(QUEUE_KEY)

def enqueue_due_executions(look_ahead_minutes: int = 5):
    """Find PENDING executions due soon and push them to Redis.
    
    This is the second responsibility of the watcher.
    """
    conn = get_db()
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT e.id, e.job_id, e.scheduled_at, j.task_id, j.parameters
        FROM executions e
        JOIN jobs j ON e.job_id = j.id
        WHERE e.status = 'PENDING'
          AND e.scheduled_at <= NOW() + INTERVAL '%s minutes'
        ORDER BY e.scheduled_at
    """, (look_ahead_minutes,))
    
    rows = cursor.fetchall()
    enqueued = 0
    
    for exec_id, job_id, scheduled_at, task_id, params in rows:
        member = json.dumps({
            "execution_id": str(exec_id),
            "job_id": str(job_id),
            "task_id": task_id,
            "parameters": params or {}
        })
        r.zadd(QUEUE_KEY, {member: scheduled_at.timestamp()})
        
        cursor.execute(
            "UPDATE executions SET status = 'QUEUED' WHERE id = %s AND status = 'PENDING'",
            (exec_id,)
        )
        enqueued += 1
    
    conn.commit()
    conn.close()
    return enqueued

# Use a large look-ahead for demo purposes
enqueued = enqueue_due_executions(look_ahead_minutes=60 * 24 * 7)
print(f"📤 Enqueued {enqueued} executions into Redis")
print(f"   Redis queue size: {r.zcard(QUEUE_KEY)}")

## 🪦 Handling Immediate & One-Time Jobs

Not every job is a cron job. We also need to handle:

- **IMMEDIATE** jobs — run as soon as possible
- **DATE** jobs — run once at a specific future time

For immediate jobs scheduled to run within the next watcher poll window (5 minutes), we should enqueue them **directly** into Redis at creation time — no need to wait for the watcher.

In [ ]:
def create_job(user_id: str, task_id: str, schedule_type: str,
               schedule_value: str = None, parameters: dict = None) -> str:
    """Create a new job and its first execution.
    
    For IMMEDIATE jobs: create execution + enqueue to Redis right away.
    For DATE jobs: create execution with the target timestamp.
    For CRON jobs: the watcher will create executions on its next poll.
    """
    conn = get_db()
    cursor = conn.cursor()
    
    job_id = str(uuid.uuid4())
    
    # Insert the job
    cursor.execute("""
        INSERT INTO jobs (id, user_id, task_id, schedule_type, schedule_value, parameters)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (job_id, user_id, task_id, schedule_type, schedule_value,
          json.dumps(parameters or {})))
    
    if schedule_type == 'IMMEDIATE':
        now = datetime.now()
        exec_id = str(uuid.uuid4())
        cursor.execute("""
            INSERT INTO executions (id, job_id, status, scheduled_at)
            VALUES (%s, %s, 'QUEUED', %s)
        """, (exec_id, job_id, now))
        
        # Enqueue directly to Redis — don't wait for the watcher
        member = json.dumps({
            "execution_id": exec_id,
            "job_id": job_id,
            "task_id": task_id,
            "parameters": parameters or {}
        })
        r.zadd(QUEUE_KEY, {member: now.timestamp()})
        print(f"  ⚡ IMMEDIATE job → enqueued to Redis directly")
        
    elif schedule_type == 'DATE':
        scheduled_at = datetime.fromisoformat(schedule_value)
        exec_id = str(uuid.uuid4())
        cursor.execute("""
            INSERT INTO executions (id, job_id, status, scheduled_at)
            VALUES (%s, %s, 'PENDING', %s)
        """, (exec_id, job_id, scheduled_at))
        print(f"  📅 DATE job → execution created for {schedule_value}")
        
    elif schedule_type == 'CRON':
        print(f"  🔄 CRON job → watcher will create executions on next poll")
    
    conn.commit()
    conn.close()
    return job_id

# Create one of each type
print("Creating jobs:")
print()

id1 = create_job("user_demo", "send_email", "IMMEDIATE",
                 parameters={"to": "urgent@example.com", "subject": "Alert!"})

id2 = create_job("user_demo", "generate_report", "DATE", "2026-05-01T10:00:00",
                 parameters={"report_type": "quarterly"})

id3 = create_job("user_demo", "send_webhook", "CRON", "0 */6 * * *",
                 parameters={"url": "https://api.example.com/health"})

print()
print(f"Redis queue size: {r.zcard(QUEUE_KEY)}")

## ☠️ Dead Letter Queue (DLQ)

When a job fails its maximum number of retries, it's **permanently failed**. We don't want to lose track of it — we need to:

1. Record it in a dead letter queue table for investigation
2. Alert operators so they can fix the root cause
3. Potentially replay the job after the fix

```
Job fails attempt 1 → retry (5s delay)
Job fails attempt 2 → retry (25s delay)
Job fails attempt 3 → retry (125s delay)
Job fails attempt 4 → ☠️ DEAD LETTER QUEUE
```

Common reasons jobs end up in the DLQ:
- **Poison pills** — malformed input that always causes a crash
- **Upstream dependency down** — the service the task calls is permanently broken
- **Bug in task code** — a code change introduced a regression

In [ ]:
# Dead Letter Queue implementation

MAX_RETRIES = 3

def send_to_dlq(execution_id: str, job_id: str, task_id: str,
                parameters: dict, error_message: str, attempts: int):
    """Move a permanently failed job to the Dead Letter Queue.
    
    This records all context needed to investigate and replay the job.
    """
    conn = get_db()
    cursor = conn.cursor()
    
    # Insert into DLQ table
    cursor.execute("""
        INSERT INTO dead_letter_queue 
            (execution_id, job_id, task_id, parameters, error_message, attempts)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (execution_id, job_id, task_id, json.dumps(parameters),
          error_message, attempts))
    
    # Update execution status to FAILED
    cursor.execute("""
        UPDATE executions 
        SET status = 'FAILED', 
            completed_at = NOW(),
            result = %s
        WHERE id = %s
    """, (json.dumps({"error": error_message, "attempts": attempts}), execution_id))
    
    conn.commit()
    conn.close()

def get_dlq_entries(limit: int = 10) -> list:
    """Retrieve entries from the Dead Letter Queue for investigation."""
    conn = get_db()
    cursor = conn.cursor(psycopg2.extras.RealDictCursor)
    
    cursor.execute("""
        SELECT dlq.id::text, dlq.task_id, dlq.error_message, 
               dlq.attempts, dlq.failed_at, dlq.parameters
        FROM dead_letter_queue dlq
        ORDER BY dlq.failed_at DESC
        LIMIT %s
    """, (limit,))
    
    entries = cursor.fetchall()
    conn.close()
    return entries

print("✅ Dead Letter Queue functions defined")

In [ ]:
# Simulate a full worker cycle with DLQ handling

PROCESSING_KEY = "scheduler:processing"
r.delete(PROCESSING_KEY)

# Simulated handlers — "send_webhook" always fails
def handle_send_email(params):
    return {"sent": True}

def handle_send_webhook_broken(params):
    """This handler is permanently broken — simulates a poison pill."""
    raise Exception("Connection refused: https://api.example.com/health")

HANDLERS = {
    "send_email": handle_send_email,
    "send_webhook": handle_send_webhook_broken,
}

def process_job_with_dlq(job_data: dict, worker_id: str):
    """Process a single job, handling retries and DLQ."""
    exec_id = job_data["execution_id"]
    task_id = job_data["task_id"]
    attempt = job_data.get("attempt", 0)
    params = job_data.get("parameters", {})
    
    handler = HANDLERS.get(task_id)
    if not handler:
        print(f"  [{worker_id}] ❓ Unknown task: {task_id}")
        return
    
    try:
        result = handler(params)
        print(f"  [{worker_id}] ✅ {exec_id[:8]}... {task_id} succeeded")
        
    except Exception as e:
        if attempt + 1 >= MAX_RETRIES:
            # Max retries exceeded → Dead Letter Queue
            send_to_dlq(
                execution_id=exec_id,
                job_id=job_data.get("job_id", exec_id),
                task_id=task_id,
                parameters=params,
                error_message=str(e),
                attempts=attempt + 1
            )
            print(f"  [{worker_id}] ☠️  {exec_id[:8]}... {task_id} → DLQ after {attempt + 1} attempts: {e}")
        else:
            # Retry with backoff
            delay = 5.0 * (5.0 ** attempt)
            retry_member = json.dumps({
                **job_data,
                "attempt": attempt + 1
            })
            r.zadd(QUEUE_KEY, {retry_member: time.time() + delay})
            print(f"  [{worker_id}] 🔄 {exec_id[:8]}... {task_id} retry #{attempt + 1} in {delay:.0f}s: {e}")

# Enqueue test jobs: one good email and one broken webhook
r.delete(QUEUE_KEY)

now = time.time()

# Good job
r.zadd(QUEUE_KEY, {json.dumps({
    "execution_id": str(uuid.uuid4()),
    "job_id": str(uuid.uuid4()),
    "task_id": "send_email",
    "parameters": {"to": "happy@example.com"},
    "attempt": 0
}): now})

# Poison pill — will always fail
poison_exec_id = str(uuid.uuid4())
poison_job_id = str(uuid.uuid4())
for attempt in range(MAX_RETRIES):
    r.zadd(QUEUE_KEY, {json.dumps({
        "execution_id": poison_exec_id,
        "job_id": poison_job_id,
        "task_id": "send_webhook",
        "parameters": {"url": "https://api.example.com/health"},
        "attempt": attempt
    }): now + attempt})

print("🏭 Processing jobs:")
print("=" * 70)

# Process all queued jobs
while True:
    results = r.zpopmin(QUEUE_KEY, count=1)
    if not results:
        break
    member, score = results[0]
    job_data = json.loads(member)
    process_job_with_dlq(job_data, "worker-1")

In [ ]:
# Inspect the Dead Letter Queue

entries = get_dlq_entries()

print("☠️  Dead Letter Queue")
print("=" * 70)

if not entries:
    print("  (empty)")
else:
    for entry in entries:
        print(f"  ID: {entry['id'][:8]}...")
        print(f"  Task: {entry['task_id']}")
        print(f"  Error: {entry['error_message']}")
        print(f"  Attempts: {entry['attempts']}")
        print(f"  Failed at: {entry['failed_at']}")
        print(f"  Parameters: {entry['parameters']}")
        print()

print("💡 DLQ entries preserve all context needed to:")
print("   1. Investigate why the job failed")
print("   2. Fix the root cause (broken endpoint, bad params)")
print("   3. Replay the job after the fix")

In [ ]:
# Replaying a job from the DLQ

def replay_from_dlq(dlq_entry_id: str):
    """Re-enqueue a job from the Dead Letter Queue for another attempt.
    
    Use this after fixing the root cause of the failure.
    """
    conn = get_db()
    cursor = conn.cursor(psycopg2.extras.RealDictCursor)
    
    cursor.execute("""
        SELECT execution_id::text, job_id::text, task_id, parameters
        FROM dead_letter_queue WHERE id = %s
    """, (dlq_entry_id,))
    
    entry = cursor.fetchone()
    if not entry:
        print(f"❌ DLQ entry {dlq_entry_id} not found")
        return
    
    # Create a fresh execution for this job
    new_exec_id = str(uuid.uuid4())
    cursor.execute("""
        INSERT INTO executions (id, job_id, status, scheduled_at)
        VALUES (%s, %s, 'QUEUED', NOW())
    """, (new_exec_id, entry['job_id']))
    
    # Enqueue to Redis
    member = json.dumps({
        "execution_id": new_exec_id,
        "job_id": entry['job_id'],
        "task_id": entry['task_id'],
        "parameters": entry['parameters'] if isinstance(entry['parameters'], dict) else json.loads(entry['parameters']),
        "attempt": 0  # fresh start
    })
    r.zadd(QUEUE_KEY, {member: time.time()})
    
    # Remove from DLQ
    cursor.execute("DELETE FROM dead_letter_queue WHERE id = %s", (dlq_entry_id,))
    
    conn.commit()
    conn.close()
    print(f"🔄 Replayed job from DLQ → new execution {new_exec_id[:8]}...")

# Replay the first DLQ entry (if any)
if entries:
    replay_from_dlq(entries[0]['id'])
    print(f"   Queue size: {r.zcard(QUEUE_KEY)}")
else:
    print("No DLQ entries to replay")

## 🏗️ Full System Architecture

Let's put it all together. Here's the complete flow:

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        JOB SCHEDULER ARCHITECTURE                       │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  ┌──────┐    POST /jobs     ┌───────────┐                              │
│  │ User │  ──────────────►  │ API Server│                              │
│  └──────┘                   └─────┬─────┘                              │
│                                   │                                     │
│                          writes   │                                     │
│                                   ▼                                     │
│                           ┌──────────────┐                              │
│                           │  PostgreSQL   │                              │
│                           │  - jobs       │◄──── Watcher polls ──┐      │
│                           │  - executions │      every ~5 min    │      │
│                           │  - DLQ        │                      │      │
│                           └──────────────┘               ┌───────┴───┐  │
│                                                          │  Watcher  │  │
│       ┌──────────────────────────────────────────────┐   │  Process  │  │
│       │              Redis                           │   └───────┬───┘  │
│       │  ┌──────────────────────────────┐           │           │      │
│       │  │ ZSET job_queue               │◄──────────────────────┘      │
│       │  │ score = scheduled_at          │           │                  │
│       │  │ member = {exec_id, task, ...} │           │                  │
│       │  └──────────────────────────────┘           │                  │
│       └──────────────┬───────────────────────────────┘                  │
│                      │                                                  │
│              workers pull                                               │
│                      │                                                  │
│          ┌───────────┼───────────┐                                      │
│          ▼           ▼           ▼                                      │
│     ┌────────┐ ┌────────┐ ┌────────┐                                   │
│     │Worker 1│ │Worker 2│ │Worker 3│                                   │
│     └───┬────┘ └───┬────┘ └───┬────┘                                   │
│         │          │          │                                         │
│         ├─ ✅ success → update DB                                       │
│         ├─ 🔄 retry → re-enqueue with backoff                          │
│         └─ ☠️ max retries → Dead Letter Queue                           │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Final summary: count everything in the system

conn = get_db()
cursor = conn.cursor()

print("📊 System Status Dashboard")
print("=" * 50)

cursor.execute("SELECT COUNT(*) FROM tasks")
print(f"  Tasks defined:     {cursor.fetchone()[0]}")

cursor.execute("SELECT COUNT(*) FROM jobs")
print(f"  Jobs scheduled:    {cursor.fetchone()[0]}")

cursor.execute("SELECT COUNT(*) FROM jobs WHERE schedule_type = 'CRON' AND is_active = TRUE")
print(f"  Active cron jobs:  {cursor.fetchone()[0]}")

print()
print("  Executions by status:")
cursor.execute("""
    SELECT status, COUNT(*) 
    FROM executions 
    GROUP BY status 
    ORDER BY COUNT(*) DESC
""")
for status, count in cursor.fetchall():
    bar = "█" * min(count, 30)
    print(f"    {status:<12} {count:>4}  {bar}")

cursor.execute("SELECT COUNT(*) FROM dead_letter_queue")
dlq_count = cursor.fetchone()[0]
print(f"\n  Dead Letter Queue: {dlq_count} entries")

print(f"\n  Redis queue size:  {r.zcard(QUEUE_KEY)}")

conn.close()

## 🧹 Cleanup

In [ ]:
# Clean up Redis keys
r = get_redis()
for key in r.keys("scheduler:*"):
    r.delete(key)
print("🧹 Cleaned up Redis keys")

# Note: database data persists for you to explore in Adminer.
# To fully reset, run: docker-compose down -v && docker-compose up -d
print("💡 Database data preserved — explore it in Adminer at http://localhost:8080")

## 📚 Summary

### Key Takeaways

1. **Cron expressions** are a compact syntax for recurring schedules — `croniter` parses them and computes next runs
2. **The watcher** runs periodically, creating execution rows for cron jobs and enqueuing due executions to Redis
3. **Immediate jobs** bypass the watcher and go straight to Redis for minimum latency
4. **Dead letter queues** capture permanently failed jobs with full context for investigation and replay
5. **The two-phase architecture** (DB → Redis → Workers) balances durability with precision

### Interview Tips

- Mention the **job/execution separation** early — it shows you understand recurring workloads
- The **two-phase watcher** (DB poll + message queue) is the key to meeting the 2-second precision requirement
- Always discuss **DLQ** when talking about failure handling — it shows production-readiness thinking
- Know the trade-offs between SQS (managed), Redis (self-hosted), and RabbitMQ (feature-rich)

### Full Series Review

| Notebook | Core Concept |
|----------|--------------|
| 1. Job Queue & Priority | Redis ZSET, fair scheduling, two-phase architecture |
| 2. Distributed Execution | Visibility timeouts, retries, idempotency, heartbeats |
| 3. Cron & DLQ (this one) | croniter, watcher process, dead letter queues, replay |